# 教材候选学习价值复核
只读取安全摘要，不读取题目、教材、答案或模型输出。两轮匿名评分不同的题保持隔离；已筛选一致子集成绩不是204题总体准确率。

In [ ]:
import json
from collections import Counter
from pathlib import Path
root=Path.cwd();root=root.parent if root.name=='docs' else root
def read(n):return json.loads((root/'docs'/n).read_text(encoding='utf-8'))
a=read('book_task_aligned_value_result_20260908.safe.json')
p=read('book_task_aligned_value_protocol_20260908.safe.json')
assert a['items']==p['items']==204
assert a['generations']==p['generations_total']==816
assert len(a['decisions'])==408 and len({(r['id'],r['model']) for r in a['decisions']})==408
assert sum(a['grade_consensus'].values())==153
for row in a['table']:
    ds=[r for r in a['decisions'] if all(r[k]==row[k] for k in ('split','model','kind'))]
    assert len(ds)==row['items'] and dict(Counter(r['decision'] for r in ds))==row['decisions']
    if row['kind']=='evidence_selection':
        for k in ('original_correct','permuted_correct','answers_changed','invalid_responses'):assert sum(r[k] for r in ds)==row[k]
    else:
        ss=[r for r in a['open_scores'] if all(r[k]==row[k] for k in ('split','model','kind'))]
        assert len(ss)==row['agreed_items']
        assert sum(r['closed_score']==2 for r in ss)==row['closed_correct']
        assert sum(r['evidence_score']==2 for r in ss)==row['evidence_correct']
assert sum(r['groups'] for r in a['pools'].values())==51
assert sum(r['questions'] for r in a['pools'].values())==204
assert a['pools']['development']['groups']==8
assert not a['training'] and not a['training_ready']
c=read('book_task_aligned_value_context_20260908.safe.json')
bad={r['id'] for r in c['flags']}
assert len(bad)==c['flagged_questions']==17
clean=[r for r in a['open_scores'] if r['id'] not in bad]
assert len({r['id'] for r in clean})==c['clean_agreed_questions']==110
for row in c['clean_aggregate']:
    rs=[r for r in clean if r['model']==row['model']]
    assert len(rs)==row['items']
    assert sum(r['closed_score']==2 for r in rs)==row['closed_correct']
    assert sum(r['evidence_score']==2 for r in rs)==row['evidence_correct']
assert sum(r['groups'] for r in c['pools'].values())==51
assert c['pools']['learning_candidate']=={'groups':8,'questions':32}
assert c['pools']['quarantine']=={'groups':35,'questions':140}
assert c['pools']['development']=={'groups':8,'questions':32}
i=read('book_task_aligned_value_pool_integrity_20260908.safe.json')
assert i['groups']==51 and i['questions']==204
assert i['duplicate_groups']==i['duplicate_questions']==i['changed_questions']==i['split_migrations']==0
for pool,counts in c['pools'].items():
    assert all(i['pools'][pool][k]==v for k,v in counts.items())
for name in ('step120','cpt'):
    s=read('book_task_aligned_value_'+name+'_20260908.safe.json')
    assert s['responses']==408 and s['cases_sha256']==p['cases_sha256']
print('PASS: 16 cells recomputed, context-filtered 110 paired items, final 8 learning/35 quarantine/8 dev groups, 816 generations')
